In [6]:
%pip install -U "segmind>=1.1.0"

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# API key removed; set SEGMIND_API_KEY in the environment instead


In [7]:
import os
import getpass

api_key = getpass.getpass("Enter your Segmind API key: ")
os.environ["SEGMIND_API_KEY"] = api_key

print("API key configured.")

API key configured.


In [11]:
from pathlib import Path
import base64
import segmind

IMAGE_FOLDER = Path(
    "/Users/wenlanzhang/Downloads/heldout_GSVI_p100_sc100pct_fp_conf001/"
)

PROMPT = """
Are there any clearly defined piles of household waste (e.g., large amounts of plastic bags, 
food packaging, paper scraps, beverage bottles, or other domestic waste)? Please answer Yes or No.
""".strip()


def image_to_data_url(image_path):
    suffix = image_path.suffix.lower()

    mime_types = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".webp": "image/webp"
    }

    mime = mime_types[suffix]

    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    return f"data:{mime};base64,{encoded}"


# Find images
images = sorted([
    p for p in IMAGE_FOLDER.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]
])

print("Images found:", len(images))

# Test first image
test_image = images[0]

print("Testing:", test_image.name)

image_data = image_to_data_url(test_image)

payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": PROMPT
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_data
                    }
                }
            ]
        }
    ]
}

reply = segmind.chat(
    "qwen2-vl-72b-instruct",
    **payload
)

print(reply.text)

Images found: 30
Testing: 0o5SkIs7bA6N8GlGtD4jKQ_90.jpg
No.


In [12]:
import csv
import time

OUTPUT_CSV = IMAGE_FOLDER / "qwen2_vl_72b_segmind_results.csv"

with open(
    OUTPUT_CSV,
    "w",
    newline="",
    encoding="utf-8"
) as csvfile:

    writer = csv.DictWriter(
        csvfile,
        fieldnames=[
            "filename",
            "filepath",
            "qwen_response",
            "status",
            "error"
        ]
    )

    writer.writeheader()

    for i, image_path in enumerate(images, start=1):

        print(
            f"[{i}/{len(images)}] "
            f"Processing {image_path.name} ..."
        )

        try:
            image_data = image_to_data_url(image_path)

            payload = {
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": PROMPT
                            },
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": image_data
                                }
                            }
                        ]
                    }
                ]
            }

            reply = segmind.chat(
                "qwen2-vl-72b-instruct",
                **payload
            )

            answer = reply.text

            writer.writerow({
                "filename": image_path.name,
                "filepath": str(image_path),
                "qwen_response": answer,
                "status": "success",
                "error": ""
            })

            csvfile.flush()

            print("    ✓ Done")

        except Exception as e:

            writer.writerow({
                "filename": image_path.name,
                "filepath": str(image_path),
                "qwen_response": "",
                "status": "error",
                "error": str(e)
            })

            csvfile.flush()

            print(f"    ✗ Error: {e}")

        time.sleep(0.5)


print()
print("Finished.")
print("Results saved to:")
print(OUTPUT_CSV)

[1/30] Processing 0o5SkIs7bA6N8GlGtD4jKQ_90.jpg ...
    ✓ Done
[2/30] Processing 1JNhbluOQCy_rje7QQfMQg_270.jpg ...
    ✓ Done
[3/30] Processing 4xRipV3paUvRC250PTMxEw_90.jpg ...
    ✓ Done
[4/30] Processing 6MVWMZr9n-bLlFxnfmiFbg_180.jpg ...
    ✓ Done
[5/30] Processing 7K1SkE88jDGHNv2c-CDDew_180.jpg ...
    ✓ Done
[6/30] Processing 7K1SkE88jDGHNv2c-CDDew_90.jpg ...
    ✓ Done
[7/30] Processing 8CRUCB9bfg-EM-HiZLq_8A_0.jpg ...
    ✓ Done
[8/30] Processing 8CRUCB9bfg-EM-HiZLq_8A_270.jpg ...
    ✓ Done
[9/30] Processing IHJKVhGoq3VPqt0QzMPEeg_0.jpg ...
    ✓ Done
[10/30] Processing JlgxfM6hoXsLAdMICMOF3A_0.jpg ...
    ✓ Done
[11/30] Processing JlgxfM6hoXsLAdMICMOF3A_90.jpg ...
    ✓ Done
[12/30] Processing MB7jt75_t2TZ0uyZyyCYiQ_0.jpg ...
    ✓ Done
[13/30] Processing MB7jt75_t2TZ0uyZyyCYiQ_90.jpg ...
    ✓ Done
[14/30] Processing OTasQrJpDH4HAke5Vm8-IQ_180.jpg ...
    ✓ Done
[15/30] Processing OTasQrJpDH4HAke5Vm8-IQ_90.jpg ...
    ✓ Done
[16/30] Processing WVymk5Lzcs9OhNFBooIo5w_270.jp